## Using **Sentiment Analysis** example here.

In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os

In [13]:
load_dotenv()

True

In [14]:
model = ChatGroq(
    model="openai/gpt-oss-120b",    
    api_key=os.getenv("GROQ_API_KEY")
)

In [15]:
class SentimentSchema(BaseModel):

    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review.")

In [16]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [17]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [18]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [19]:
def find_sentiment(state: ReviewState):

    prompt = f"For the following review find out the sentiment \n {state["review"]}"
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment' : sentiment}

In [20]:
def positive_response(state: ReviewState):
    
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n'{state['review']}\n'
    Also, kindly ask user to leave feedback on our website.
    """
    response = model.invoke(prompt)

    return {'response':response}

In [21]:
def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review: \n\n{state['review']}\n
Return issue_type, tone and urgency.
"""

    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

In [22]:
def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

In [23]:
def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [25]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [26]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. "
    "I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'},
 'response': 'Hi\u202f[Customer Name],\n\nI’m really sorry you’re experiencing this issue – I can understand how frustrating it must be, especially when things aren’t working as they should. Thank you for bringing it to our attention and for letting us know how urgent it is.\n\n**What’s happening:**  \nYou reported a bug with **[brief description of the problem, e.g., “the checkout button not responding on the mobile app”**].  \n\n**What we’re doing right now:**  \n1. **Escalated to our engineering team** – because of the high priority, it’s already in the queue for immediate investigation.  \n2. **Assigned a dedicated specialist** – I’ll be your p